# Observability Setup Example



1. First, load environment variables from a `.env` file to configure observability settings. 

2. Second, set up observability using the `setup_observability` function from the `agent_framework.observability` module.

3. Third, demonstrate how to use the loaded environment variables in your application.

4. Fourth, check AI Foundry and the local Aspire dashboard for telemetry data.



##### Powershell command to open port for OTLP on a remote machine
`New-NetFirewallRule -DisplayName "Allow Port 4317" -Direction Inbound -LocalPort 4317 -Protocol TCP -Action Allow`

##### Docker command to run Aspire OTLP collector locally
`docker run --rm -it -p 18888:18888 -p 4317:18889 -d --name aspire-dashboard mcr.microsoft.com/dotnet/aspire-dashboard:9.5`

In [ ]:
import os
from dotenv import load_dotenv

from agent_framework.observability import setup_observability
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter

from agent_framework.observability import setup_observability
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import ConsoleSpanExporter

load_dotenv()


AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

ENABLE_OTEL = os.getenv("ENABLE_OTEL", "false").lower() == "true"
ENABLE_SENSITIVE_DATA = os.getenv("ENABLE_SENSITIVE_DATA", "false").lower() == "true"
OTLP_ENDPOINT = os.getenv("OTLP_ENDPOINT", "http://localhost:4317")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING", "")


custom_exporters = [
    OTLPSpanExporter(endpoint=OTLP_ENDPOINT),
    ConsoleSpanExporter()
]


# Enable Agent Framework telemetry with console output (default behavior)
setup_observability(enable_sensitive_data=ENABLE_SENSITIVE_DATA, 
                    otlp_endpoint=OTLP_ENDPOINT,
                    applicationinsights_connection_string=APPLICATIONINSIGHTS_CONNECTION_STRING,
                    exporters=custom_exporters
                )

# Example usage of the loaded environment variables
print("Azure OpenAI API Key:", AZURE_OPENAI_API_KEY)
print("Azure OpenAI Endpoint:", AZURE_OPENAI_ENDPOINT)
print("Enable OTEL:", ENABLE_OTEL)
print("Enable Sensitive Data:", ENABLE_SENSITIVE_DATA)
print("OTLP Endpoint:", OTLP_ENDPOINT)
print("Application Insights Connection String:", APPLICATIONINSIGHTS_CONNECTION_STRING)


### Agent Example

A simple MS Agent Framework agent example to demonstrate observability integration.




In [ ]:
import asyncio
from agent_framework.azure import AzureOpenAIChatClient
from typing import Annotated
from pydantic import Field
from agent_framework import ai_function

class WeatherTools:
    def __init__(self):
        self.last_location = None

    def get_weather(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."

    def get_weather_details(self) -> int:
        """Get the detailed weather for the last requested location."""
        if self.last_location is None:
            return "No location specified yet."
        return f"The detailed weather in {self.last_location} is cloudy with a high of 15°C, low of 7°C, and 60% humidity."
    

tools = WeatherTools()
agent = AzureOpenAIChatClient(api_key=AZURE_OPENAI_API_KEY,
    endpoint=AZURE_OPENAI_ENDPOINT,
    deployment_name="gpt-5-nano",
    ).create_agent(
    name="WeatherAgent",
    instructions="You are a helpful agent that can fetch the weather predictions.",
    tools=[tools.get_weather, tools.get_weather_details]
)

thread1 = agent.get_new_thread()

result = await agent.run("What is the weather like in Amsterdam?", thread=thread1)
print(result.text)    
result = await agent.run("Get me more details.", thread=thread1)
print(result.text)    
